## Disclaimer: This notebook was constructed with assistance from Gemini 3.1 Pro for code completion, debugging, and graph styling.

In [1]:
import os, json, re, difflib
import numpy as np
import pandas as pd
VARIANT_A_BASE_DIR = 'multimodal/variant_a'
VARIANT_B_BASE_DIR = 'multimodal/variant_b'
SCENE_TYPE_MAP = {
    'single/solo': 'Isolated', 'single_solo': 'Isolated',
    'single/multi': 'Clustered', 'single_multi': 'Clustered',
    'multi': 'Mixed',
}
SCENE_ORDER = ['Isolated', 'Clustered', 'Mixed']

def _has_json_array(raw):
    """True if the raw string contains [...] at all."""
    if not isinstance(raw, str) or not raw.strip():
        return False
    return '[' in raw and ']' in raw

def _extract_json_array(raw):
    if not isinstance(raw, str):
        return []
    a, b = raw.find('['), raw.rfind(']')
    if a == -1 or b == -1 or b < a:
        return []
    try:
        return json.loads(raw[a:b+1])
    except Exception:
        return []

def classify_zero_iou(raw):
    """Classify an IoU=0 response into rejection sub-type.
    
    Categories:
      rejection_no_json  -- no JSON array structure at all (true refusal / free-text)
      rejection_empty    -- well-formed empty array [] (cooperative null response)
      truncated          -- starts a JSON array but never closes it properly
                           (generation limit hit mid-response)
      misprediction      -- valid parsed predictions exist, but all had IoU=0
    """
    if not isinstance(raw, str) or not raw.strip():
        return 'rejection_no_json'
    
    first_bracket = raw.find('[')
    
    # No opening bracket at all -> true refusal
    if first_bracket == -1:
        return 'rejection_no_json'
    
    # Has '[' -- now check if the OUTER array closes properly.
    # Find the matching ']' by counting bracket depth from the first '['.
    depth = 0
    outer_close = -1
    for i in range(first_bracket, len(raw)):
        if raw[i] == '[':
            depth += 1
        elif raw[i] == ']':
            depth -= 1
            if depth == 0:
                outer_close = i
                break
    
    if outer_close == -1:
        # Opened '[' but never closed at depth 0 -> truncated
        return 'truncated'
    
    # Outer array closes -- try to parse it
    try:
        parsed = json.loads(raw[first_bracket:outer_close+1])
    except Exception:
        # Closes but still malformed (rare) -- treat as truncated
        return 'truncated'
    
    if not isinstance(parsed, list):
        return 'truncated'
    
    if len(parsed) == 0:
        return 'rejection_empty'
    
    # Has parsed objects model answered, predictions were just wrong
    return 'misprediction'

records = []
for variant_label, base_dir in [('Variant A', VARIANT_A_BASE_DIR),
                                 ('Variant B', VARIANT_B_BASE_DIR)]:
    for rel_dir, scene_label in SCENE_TYPE_MAP.items():
        group_dir = os.path.join(base_dir, rel_dir)
        if not os.path.isdir(group_dir):
            continue
        for fid in sorted(os.listdir(group_dir)):
            pf_path = os.path.join(group_dir, fid, 'pareto_front.json')
            if not os.path.isfile(pf_path):
                continue
            with open(pf_path) as f:
                pf = json.load(f)

            swad_idx = pf['swad_star_index']
            sol      = pf['solutions'][swad_idx]
            raw      = sol['vlm_output'].get('raw_response', '')
            iou      = sol['objectives']['iou']

            if iou > 0:
                rej_class = 'accepted'
            else:
                rej_class = classify_zero_iou(raw)

            records.append({
                'variant':          variant_label,
                'scene_type':       scene_label,
                'sample_id':        int(fid),
                'iou_m':            iou,
                'rejection_class':  rej_class,
                'has_json':         _has_json_array(raw),
                'n_parsed_objects': len(_extract_json_array(raw)),
            })

df_rej = pd.DataFrame(records)

CLASS_ORDER = ['accepted', 'rejection_no_json', 'rejection_empty',
               'truncated', 'misprediction']
CLASS_NICE  = {
    'accepted':           'Accepted (IoU > 0)',
    'rejection_no_json':  'Refused -- no JSON array (true rejection)',
    'rejection_empty':    'Returned empty [] (cooperative null)',
    'truncated':          'Truncated mid-JSON (gen-limit hit)',
    'misprediction':      'Valid predictions, all wrong (IoU = 0)',
}

print('=' * 80)
print('  REJECTION CLASSIFICATION -- SWAD*-OPTIMAL SOLUTIONS')
print('=' * 80)

for variant in ['Variant A', 'Variant B']:
    sub = df_rej[df_rej.variant == variant]
    n   = len(sub)
    n_zero = (sub.iou_m == 0).sum()
    print(f'\n  {variant}  (N={n}, IoU=0 count={n_zero})')
    print(f'  {"Class":<40} {"Count":>6} {"% of all":>9} {"% of IoU=0":>11}')
    print(f'  {"-"*68}')
    for cls in CLASS_ORDER:
        cnt = (sub.rejection_class == cls).sum()
        pct_all  = 100 * cnt / n if n else 0
        pct_zero = 100 * cnt / n_zero if n_zero and cls != 'accepted' else 0
        pz_str   = f'{pct_zero:>10.1f}%' if cls != 'accepted' else f'{"--":>11}'
        print(f'  {CLASS_NICE[cls]:<40} {cnt:>6} {pct_all:>8.1f}% {pz_str}')

    print(f'\n  Per scene type (true rejections = no_json + empty_json):')
    print(f'  {"Scene":<15} {"N":>5} {"IoU=0":>6} {"Rejected":>9} {"Rej%":>7} {"Mispred":>8}')
    print(f'  {"-"*52}')
    for st in SCENE_ORDER:
        st_sub  = sub[sub.scene_type == st]
        n_st    = len(st_sub)
        n_zero  = (st_sub.iou_m == 0).sum()
        n_rej   = st_sub.rejection_class.isin(
                      ['rejection_no_json', 'rejection_empty']).sum()
        n_mis   = (st_sub.rejection_class == 'misprediction').sum()
        print(f'  {st:<15} {n_st:>5} {n_zero:>6} {n_rej:>9} '
              f'{100*n_rej/n_st:>6.1f}% {n_mis:>8}')

print('\n' + '=' * 80)

  REJECTION CLASSIFICATION -- SWAD*-OPTIMAL SOLUTIONS

  Variant A  (N=750, IoU=0 count=349)
  Class                                     Count  % of all  % of IoU=0
  --------------------------------------------------------------------
  Accepted (IoU > 0)                          401     53.5%          --
  Refused -- no JSON array (true rejection)     85     11.3%       24.4%
  Returned empty [] (cooperative null)         19      2.5%        5.4%
  Truncated mid-JSON (gen-limit hit)           42      5.6%       12.0%
  Valid predictions, all wrong (IoU = 0)      203     27.1%       58.2%

  Per scene type (true rejections = no_json + empty_json):
  Scene               N  IoU=0  Rejected    Rej%  Mispred
  ----------------------------------------------------
  Isolated          250     83        32   12.8%       49
  Clustered         250     86        42   16.8%       34
  Mixed             250    180        30   12.0%      120

  Variant B  (N=750, IoU=0 count=509)
  Class          

In [2]:
VISUAL_RETAINED  = {'jpeg_filter','pixelate','defocus_blur',
                    'motion_blur','gaussian_noise','fog_filter'}
TEXTUAL_RETAINED = {'fragmentation','character_noise','ata_saliency',
                    'homophone','synonym'}

BASELINE_CSV = 'evaluation/baseline_iou_all.csv'
VISUAL_CSV   = 'evaluation/visual/unimodal_visual_flat.csv'
TEXTUAL_CSV  = 'evaluation/textual/unimodal_textual_flat.csv'
HUMAN_JSON   = 'evaluation/bbox_results.json'

TRUE_DENIAL = {'rejection_no_json', 'rejection_empty'}

df_bl = pd.read_csv(BASELINE_CSV)
bl_zero = (df_bl['iou'] == 0).sum()

df_vis = pd.read_csv(VISUAL_CSV)
df_vis_s4 = df_vis[(df_vis.severity == 4) &
                   (df_vis.corruption.isin(VISUAL_RETAINED))]
vis_total = len(df_vis_s4)
vis_zero  = (df_vis_s4.IoU == 0).sum()
vis_per_sample = df_vis_s4.groupby(['scene_type','sample_id'])['IoU'].min()
vis_sample_zero = (vis_per_sample == 0).sum()

df_txt = pd.read_csv(TEXTUAL_CSV)
df_txt_s4 = df_txt[(df_txt.severity == 4) &
                   (df_txt.corruption.isin(TEXTUAL_RETAINED))]
txt_total = len(df_txt_s4)
txt_zero  = (df_txt_s4.IoU == 0).sum()
txt_per_sample = df_txt_s4.groupby(['scene_type','sample_id'])['IoU'].min()
txt_sample_zero = (txt_per_sample == 0).sum()

va_sub = df_rej[df_rej.variant == 'Variant A']
vb_sub = df_rej[df_rej.variant == 'Variant B']
va_denial = va_sub.rejection_class.isin(TRUE_DENIAL).sum()
vb_denial = vb_sub.rejection_class.isin(TRUE_DENIAL).sum()
va_trunc  = (va_sub.rejection_class == 'truncated').sum()
vb_trunc  = (vb_sub.rejection_class == 'truncated').sum()

with open(HUMAN_JSON) as f:
    bbox_results = json.load(f)

human_records = []
for filename, img_data in bbox_results.items():
    mapping = img_data['internal_mapping']
    parts   = mapping.rsplit('/', 1)
    cat_path, fid = parts[0], parts[1]
    st = SCENE_TYPE_MAP.get(cat_path, 'Unknown')
    for ann in img_data['annotations']:
        for lbl in ann['labels']:
            rt = lbl['response_type']
            rr = lbl.get('reject_reason', None)
            if rt == 'reject' and rr not in ('unclear_image', 'unclear_label'):
                rt = 'skip'
            if rt == 'skip':
                continue
            human_records.append({
                'filename': filename, 'folder_id': fid,
                'scene_type': st, 'session_id': ann['session'],
                'method': ann['method'], 'variant': ann['variant'],
                'response_type': rt,
                'reject_reason': rr if rt == 'reject' else None,
            })
df_human = pd.DataFrame(human_records)
human_total  = len(df_human)
human_reject = (df_human.response_type == 'reject').sum()

print('=' * 85)
print('  ABSOLUTE DENIAL / ZERO-IoU COUNTS ACROSS CONDITIONS')
print('=' * 85)
print(f'\n  {"Condition":<38} {"Denom":>10} {"Denied":>8} {"Rate":>8}')
print(f'  {"-"*66}')

rows_tbl = [
    ('Baseline (clean)',                len(df_bl),    bl_zero),
    ('Visual  S4 (per evaluation)',     vis_total,     vis_zero),
    ('Visual  S4 (per sample, worst)',  750,           vis_sample_zero),
    ('Textual S4 (per evaluation)',     txt_total,     txt_zero),
    ('Textual S4 (per sample, worst)',  750,           txt_sample_zero),
    ('Variant A  SWAD* (true denial)',  len(va_sub),   va_denial),
    ('Variant A  SWAD* (+ truncated)',  len(va_sub),   va_denial + va_trunc),
    ('Variant B  SWAD* (true denial)',  len(vb_sub),   vb_denial),
    ('Variant B  SWAD* (+ truncated)',  len(vb_sub),   vb_denial + vb_trunc),
    ('Human (label-level reject)',      human_total,   human_reject),
]
for label, denom, denied in rows_tbl:
    rate = 100 * denied / denom if denom else 0
    print(f'  {label:<38} {denom:>10,} {denied:>8,} {rate:>7.1f}%')

print(f'\n  {"-"*66}')
print(f'\n  Visual S4 per corruption (IoU=0 / 750):')
for c in sorted(VISUAL_RETAINED):
    sub = df_vis_s4[df_vis_s4.corruption == c]
    n_z = (sub.IoU == 0).sum()
    print(f'    {c:<20} {n_z:>5} / {len(sub):>5}  ({100*n_z/len(sub):>5.1f}%)')

print(f'\n  Textual S4 per corruption (IoU=0 / 750):')
for c in sorted(TEXTUAL_RETAINED):
    sub = df_txt_s4[df_txt_s4.corruption == c]
    n_z = (sub.IoU == 0).sum()
    print(f'    {c:<20} {n_z:>5} / {len(sub):>5}  ({100*n_z/len(sub):>5.1f}%)')

print(f'\n  {"-"*66}')
print(f'\n  By scene type (denial rate):')
print(f'  {"Scene":<12} {"Baseline":>9} {"Vis S4":>9} {"Txt S4":>9} '
      f'{"Var A":>9} {"Var B":>9} {"Human":>9}')
print(f'  {"-"*60}')

def _pct(num, den):
    return f'{100*num/den:.1f}%' if den else 'N/A'

for st in SCENE_ORDER:
    bl_z = (df_bl[df_bl.scene_type == st]['iou'] == 0).sum()
    bl_n = len(df_bl[df_bl.scene_type == st])

    vs = df_vis_s4[df_vis_s4.scene_type == st]
    vs_worst = vs.groupby('sample_id')['IoU'].min()
    vs_z = (vs_worst == 0).sum(); vs_n = len(vs_worst)

    ts = df_txt_s4[df_txt_s4.scene_type == st]
    ts_worst = ts.groupby('sample_id')['IoU'].min()
    ts_z = (ts_worst == 0).sum(); ts_n = len(ts_worst)

    va = va_sub[va_sub.scene_type == st]
    va_z = va.rejection_class.isin(TRUE_DENIAL).sum()
    vb = vb_sub[vb_sub.scene_type == st]
    vb_z = vb.rejection_class.isin(TRUE_DENIAL).sum()

    hs = df_human[df_human.scene_type == st]
    hs_z = (hs.response_type == 'reject').sum()

    print(f'  {st:<12} {_pct(bl_z,bl_n):>9} {_pct(vs_z,vs_n):>9} '
          f'{_pct(ts_z,ts_n):>9} {_pct(va_z,len(va)):>9} '
          f'{_pct(vb_z,len(vb)):>9} {_pct(hs_z,len(hs)):>9}')

print()

  ABSOLUTE DENIAL / ZERO-IoU COUNTS ACROSS CONDITIONS

  Condition                                   Denom   Denied     Rate
  ------------------------------------------------------------------
  Baseline (clean)                              750        0     0.0%
  Visual  S4 (per evaluation)                 4,500       35     0.8%
  Visual  S4 (per sample, worst)                750       20     2.7%
  Textual S4 (per evaluation)                 3,750      207     5.5%
  Textual S4 (per sample, worst)                750      163    21.7%
  Variant A  SWAD* (true denial)                750      104    13.9%
  Variant A  SWAD* (+ truncated)                750      146    19.5%
  Variant B  SWAD* (true denial)                750      151    20.1%
  Variant B  SWAD* (+ truncated)                750      209    27.9%
  Human (label-level reject)                    663      161    24.3%

  ------------------------------------------------------------------

  Visual S4 per corruption (IoU=0 /

In [3]:
RESULT_FILE_MAP = {'l2': 'best_l2_result.json', 'swad': 'best_swad_result.json'}

def _norm(t):
    if not isinstance(t, str): return ''
    return re.sub(r'[\s\u200b\u200d]+', '', t).lower()

def _labels_match(a, b):
    na, nb = _norm(a), _norm(b)
    if na == nb: return True
    if sorted(na) == sorted(nb): return True
    return difflib.SequenceMatcher(None, na, nb).ratio() >= 0.75

def _calc_iou(a, b):
    xa, ya = max(a[0],b[0]), max(a[1],b[1])
    xb, yb = min(a[2],b[2]), min(a[3],b[3])
    inter  = max(0, xb-xa) * max(0, yb-ya)
    area   = (max(0,a[2]-a[0])*max(0,a[3]-a[1]) +
              max(0,b[2]-b[0])*max(0,b[3]-b[1]) - inter)
    return inter / area if area > 0 else 0.0

def _extract_prompt_labels(prompt):
    m = re.search(r'objects\s+"(.*?)"', prompt)
    if not m: m = re.search(r'"([^"]+)"', prompt)
    if not m: return []
    return [l.strip() for l in m.group(1).split(',')]

sample_manifest = {}

for filename, img_data in bbox_results.items():
    mapping = img_data['internal_mapping']
    parts   = mapping.rsplit('/', 1)
    cat_path, fid = parts[0], parts[1]
    st  = SCENE_TYPE_MAP.get(cat_path, 'Unknown')
    sid = int(fid)
    key = (st, sid)

    if key not in sample_manifest:
        sample_manifest[key] = {
            'scene_type': st, 'sample_id': sid,
            'n_labels': 0, 'n_reject': 0, 'n_bbox': 0,
        }

    for ann in img_data['annotations']:
        for lbl in ann['labels']:
            rt = lbl['response_type']
            rr = lbl.get('reject_reason', None)
            if rt == 'reject' and rr not in ('unclear_image', 'unclear_label'):
                continue  # skip
            sample_manifest[key]['n_labels'] += 1
            if rt == 'reject':
                sample_manifest[key]['n_reject'] += 1
            else:
                sample_manifest[key]['n_bbox'] += 1

for key in sample_manifest:
    m = sample_manifest[key]
    if m['n_reject'] == 0:
        m['rejection_status'] = 'fully_accepted'
    elif m['n_bbox'] == 0:
        m['rejection_status'] = 'full_rejection'
    else:
        m['rejection_status'] = 'partial_rejection'

df_manifest = pd.DataFrame(sample_manifest.values())

n_full_rej    = (df_manifest.rejection_status == 'full_rejection').sum()
n_partial_rej = (df_manifest.rejection_status == 'partial_rejection').sum()
n_accepted    = (df_manifest.rejection_status == 'fully_accepted').sum()

print(f'Human eval covers {len(df_manifest)} unique samples')
print(f'  Full rejection (excluded):    {n_full_rej}')
print(f'  Partial rejection (excluded): {n_partial_rej}')
print(f'  Fully accepted (kept):        {n_accepted}')
print(f'\nBy scene type:')
print(f'  {"Scene":<12} {"Full rej":>9} {"Partial":>9} {"Accepted":>9} {"Total":>7}')
print(f'  {"-"*48}')
for st in SCENE_ORDER:
    sub = df_manifest[df_manifest.scene_type == st]
    print(f'  {st:<12} '
          f'{(sub.rejection_status=="full_rejection").sum():>9} '
          f'{(sub.rejection_status=="partial_rejection").sum():>9} '
          f'{(sub.rejection_status=="fully_accepted").sum():>9} '
          f'{len(sub):>7}')

human_iou_records = []

for filename, img_data in bbox_results.items():
    mapping = img_data['internal_mapping']
    parts   = mapping.rsplit('/', 1)
    cat_path, fid = parts[0], parts[1]
    st  = SCENE_TYPE_MAP.get(cat_path, 'Unknown')
    sid = int(fid)

    for ann in img_data['annotations']:
        variant = ann['variant']
        method  = ann['method']

        base = VARIANT_A_BASE_DIR if variant == 'variant_a' else VARIANT_B_BASE_DIR
        res_path = os.path.join(base, cat_path, fid, RESULT_FILE_MAP[method])
        if not os.path.isfile(res_path):
            continue
        with open(res_path) as f:
            result = json.load(f)

        gt     = result.get('ground_truth_bboxes', {})
        orig_p = _extract_prompt_labels(result.get('original_prompt', ''))
        corr_p = _extract_prompt_labels(
                     result.get('vlm_output', {}).get('adversarial_prompt', ''))

        for lbl_entry in ann['labels']:
            rt = lbl_entry['response_type']
            rr = lbl_entry.get('reject_reason', None)
            if rt == 'reject' and rr not in ('unclear_image', 'unclear_label'):
                continue  # skip
            if rt == 'reject':
                human_iou_records.append({
                    'scene_type': st, 'sample_id': sid, 'human_iou': 0.0})
                continue

            cl = lbl_entry['label']
            target_orig = None
            for i, c in enumerate(corr_p):
                if _labels_match(cl, c) and i < len(orig_p):
                    target_orig = orig_p[i]; break

            matched_gt = []
            if gt:
                for k, v in gt.items():
                    gc = k.rsplit('_', 1)[0] if '_' in k else k
                    if (target_orig and _labels_match(gc, target_orig)) or \
                       _labels_match(gc, cl):
                        matched_gt.append(
                            [v['xmin'], v['ymin'], v['xmax'], v['ymax']])
                if not matched_gt and len(gt) == 1:
                    for v in gt.values():
                        matched_gt.append(
                            [v['xmin'], v['ymin'], v['xmax'], v['ymax']])

            hb = lbl_entry.get('bboxes', [])
            if not hb or not matched_gt:
                human_iou_records.append({
                    'scene_type': st, 'sample_id': sid, 'human_iou': 0.0})
                continue

            hbc  = [[b['xmin'],b['ymin'],b['xmax'],b['ymax']] for b in hb]
            ious = [max((_calc_iou(g, h) for h in hbc), default=0.0)
                    for g in matched_gt]
            human_iou_records.append({
                'scene_type': st, 'sample_id': sid,
                'human_iou': np.mean(ious) if ious else 0.0,
            })

df_hiou = pd.DataFrame(human_iou_records)
df_hiou_sample = df_hiou.groupby(['scene_type','sample_id'])['human_iou'] \
                        .mean().reset_index()

non_denied = df_manifest[df_manifest.rejection_status == 'fully_accepted'] \
             [['scene_type','sample_id']].copy()
partial    = df_manifest[df_manifest.rejection_status == 'partial_rejection'] \
             [['scene_type','sample_id']].copy()
full_rej   = df_manifest[df_manifest.rejection_status == 'full_rejection'] \
             [['scene_type','sample_id']].copy()

print(f'\nNon-denied by scene type:')
print(non_denied.groupby('scene_type').size().reindex(SCENE_ORDER))

df_bl_m = pd.read_csv(BASELINE_CSV).rename(columns={'iou': 'baseline_iou'})

df_vis_m = pd.read_csv(VISUAL_CSV)
vis_worst = df_vis_m[(df_vis_m.severity == 4) &
                     (df_vis_m.corruption.isin(VISUAL_RETAINED))] \
            .groupby(['scene_type','sample_id'])['IoU'].min() \
            .reset_index().rename(columns={'IoU': 'visual_worst_s4'})

df_txt_m = pd.read_csv(TEXTUAL_CSV)
txt_worst = df_txt_m[(df_txt_m.severity == 4) &
                     (df_txt_m.corruption.isin(TEXTUAL_RETAINED))] \
            .groupby(['scene_type','sample_id'])['IoU'].min() \
            .reset_index().rename(columns={'IoU': 'textual_worst_s4'})

va_iou = df_rej[df_rej.variant == 'Variant A'] \
         [['scene_type','sample_id','iou_m']] \
         .rename(columns={'iou_m': 'variant_a_iou'})
vb_iou = df_rej[df_rej.variant == 'Variant B'] \
         [['scene_type','sample_id','iou_m']] \
         .rename(columns={'iou_m': 'variant_b_iou'})

def build_comparison(sample_set):
    c = sample_set.merge(df_bl_m[['scene_type','sample_id','baseline_iou']],
                         on=['scene_type','sample_id'], how='left')
    c = c.merge(vis_worst,      on=['scene_type','sample_id'], how='left')
    c = c.merge(txt_worst,      on=['scene_type','sample_id'], how='left')
    c = c.merge(va_iou,         on=['scene_type','sample_id'], how='left')
    c = c.merge(vb_iou,         on=['scene_type','sample_id'], how='left')
    c = c.merge(df_hiou_sample, on=['scene_type','sample_id'], how='left')
    return c

comp_ok      = build_comparison(non_denied)
comp_partial = build_comparison(partial)
comp_full    = build_comparison(full_rej)

print(f'\nNon-denied merge: {len(comp_ok)} rows, '
      f'Baseline={comp_ok.baseline_iou.notna().sum()}, '
      f'Visual={comp_ok.visual_worst_s4.notna().sum()}, '
      f'Textual={comp_ok.textual_worst_s4.notna().sum()}, '
      f'VarA={comp_ok.variant_a_iou.notna().sum()}, '
      f'VarB={comp_ok.variant_b_iou.notna().sum()}, '
      f'Human={comp_ok.human_iou.notna().sum()}')

COLS     = ['baseline_iou', 'visual_worst_s4', 'textual_worst_s4',
            'variant_a_iou', 'variant_b_iou', 'human_iou']
COL_NICE = ['Baseline', 'Vis S4 worst', 'Txt S4 worst',
            'Variant A', 'Variant B', 'Human']

def print_table(df, title):
    print(f'\n  {title}')
    print(f'  {"Scene":<12} {"N":>4}', end='')
    for nice in COL_NICE:
        print(f'  {nice:>14}', end='')
    print()
    print(f'  {"-"*102}')

    for st in SCENE_ORDER + ['Overall']:
        sub = df if st == 'Overall' else df[df.scene_type == st]
        n = len(sub)
        print(f'  {st:<12} {n:>4}', end='')
        for col in COLS:
            vals = sub[col].dropna()
            if len(vals) > 0:
                print(f'  {vals.mean():>7.4f}±{vals.std():.3f}', end='')
            else:
                print(f'  {"N/A":>14}', end='')
        print()

print('\n' + '=' * 110)
print('  MEAN IoU BY CONDITION -- STRATIFIED BY HUMAN REJECTION STATUS')
print('=' * 110)
print_table(comp_ok,      'FULLY ACCEPTED -- no annotator rejected any label:')
print(f'\n  {"-"*102}')
print_table(comp_partial, 'PARTIAL REJECTION -- ≥1 annotator rejected, but ≥1 drew bboxes:')
print(f'\n  {"-"*102}')
print_table(comp_full,    'FULL REJECTION -- unanimous: all annotators rejected all labels:')
print('\n' + '=' * 110)

Human eval covers 312 unique samples
  Full rejection (excluded):    42
  Partial rejection (excluded): 69
  Fully accepted (kept):        201

By scene type:
  Scene         Full rej   Partial  Accepted   Total
  ------------------------------------------------
  Isolated            20        11        89     120
  Clustered           20        17        80     117
  Mixed                2        41        32      75

Non-denied by scene type:
scene_type
Isolated     89
Clustered    80
Mixed        32
dtype: int64

Non-denied merge: 201 rows, Baseline=201, Visual=201, Textual=201, VarA=201, VarB=201, Human=201

  MEAN IoU BY CONDITION -- STRATIFIED BY HUMAN REJECTION STATUS

  FULLY ACCEPTED -- no annotator rejected any label:
  Scene           N        Baseline    Vis S4 worst    Txt S4 worst       Variant A       Variant B           Human
  ------------------------------------------------------------------------------------------------------
  Isolated       89   0.9402±0.040   0.74